# Biohub — learned graph baseline

This notebook uses the public Biohub Tracking Support Pack's UNet + temporal edge model. It runs offline on CPU because the hosted P100 image is incompatible with Kaggle's current CUDA PyTorch build, then converts the predicted GEFF graphs into the competition's combined node/edge CSV.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUTS = Path('/kaggle/input')
support_candidates = [INPUTS / 'biohub-tracking-support-pack-50ep-v1', INPUTS / 'datasets' / 'pilkwang' / 'biohub-tracking-support-pack-50ep-v1']
SUPPORT = next(p for p in support_candidates if p.exists())
COMPETITION = INPUTS / 'competitions' / 'biohub-cell-tracking-during-development'
TEST = COMPETITION / 'test'
WORK_REPO = Path('/kaggle/working/biohub_strong_repo')
assert TEST.exists(), TEST

WHEELS = SUPPORT / 'wheels'
packages = ['tracksdata', 'zarr>=3.0.10,<4', 'pyscipopt', 'geff', 'ilpy', 'polars', 'polars-runtime-32', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy', 'donfig', 'numcodecs', 'google-crc32c', 'packaging', 'typing-extensions', 'deprecated', 'wrapt', 'geff-spec', 'bidict', 'networkx', 'click', 'cloudpickle', 'fsspec', 'tqdm', 'rich', 'pydantic', 'pydantic-core', 'typing-inspection', 'annotated-types', 'annotated-doc', 'typer', 'shellingham', 'mdurl', 'markdown-it-py', 'pygments', 'pyyaml', 'requests', 'certifi', 'charset-normalizer', 'idna', 'urllib3', 'locket', 'partd', 'toolz', 'msgpack', 'ndindex', 'psygnal', 'imageio', 'lazy-loader', 'pillow', 'tifffile', 'threadpoolctl', 'greenlet', 'numexpr']
# Preserve Kaggle's prebuilt NumPy/SciPy ABI; replacing NumPy with the
# support-pack wheel can break SciPy during GEFF conversion.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', '--find-links', str(WHEELS), *packages])
if WORK_REPO.exists(): shutil.rmtree(WORK_REPO)
shutil.copytree(SUPPORT / 'repo', WORK_REPO)
shutil.copytree(SUPPORT / 'weights', WORK_REPO / 'weights')
print('test stores:', sorted(p.name for p in TEST.glob('*.zarr')))

In [ ]:
# Run the public learned model once per test store. CPU is intentional.
script = WORK_REPO / 'scripts' / 'predict_unet_transformer.py'
weights = WORK_REPO / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
run_env = os.environ.copy()
run_env['CUDA_VISIBLE_DEVICES'] = ''
run_env['PYTHONPATH'] = str(WORK_REPO / 'src') + os.pathsep + str(WORK_REPO / 'scripts')
for video in sorted(TEST.glob('*.zarr')):
    cmd = [sys.executable, str(script), '--debug-video', str(video), '--det-threshold', '0.99', '--weights', str(weights)]
    print('running', video.name, flush=True)
    subprocess.check_call(cmd, cwd=WORK_REPO, env=run_env)

PRED_DIR = WORK_REPO / 'predictions' / os.environ.get('USER', 'root') / 'unet_transformer' / 'split_0'
if not PRED_DIR.exists():
    candidates = list((WORK_REPO / 'predictions').rglob('*.geff'))
    assert candidates, 'No GEFF predictions were produced.'
    PRED_DIR = candidates[0].parent
print('prediction files:', sorted(p.name for p in PRED_DIR.glob('*.geff')))

In [ ]:
import numpy as np
import pandas as pd
import tracksdata as td

def pick(row, names, default=-1):
    for name in names:
        if name in row:
            return row[name]
    return default

rows = []
for geff_path in sorted(PRED_DIR.glob('*.geff')):
    dataset = geff_path.stem
    loaded = td.graph.IndexedRXGraph.from_geff(geff_path)
    graph = loaded[0] if isinstance(loaded, tuple) else loaded
    node_rows = graph.node_attrs().to_dicts()
    edge_rows = graph.edge_attrs().to_dicts()
    for row in node_rows:
        node_id = pick(row, ['node_id', 'id'])
        rows.append({'dataset': dataset, 'row_type': 'node', 'node_id': int(node_id), 't': int(pick(row, ['t'])), 'z': int(round(pick(row, ['z']))), 'y': int(round(pick(row, ['y']))), 'x': int(round(pick(row, ['x']))), 'source_id': -1, 'target_id': -1})
    for row in edge_rows:
        rows.append({'dataset': dataset, 'row_type': 'edge', 'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1, 'source_id': int(pick(row, ['source_id', 'source'])), 'target_id': int(pick(row, ['target_id', 'target']))})

columns = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
submission = pd.DataFrame(rows, columns=columns)
submission.insert(0, 'id', np.arange(len(submission), dtype=np.int64))
for col in ['id', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']:
    submission[col] = submission[col].astype('int64')
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('rows:', len(submission), 'nodes:', (submission.row_type == 'node').sum(), 'edges:', (submission.row_type == 'edge').sum())
display(submission.head())